# Discover a recruitment model from the AI Act

You are building a recruitment application. Which roles and relationships does the source
support? Discover them from the complete pinned English Official Journal PDF, then inspect the
evidence behind each draft. All proposals stay pending for review.

From a clone, install the notebook environment.

```sh
uv sync --locked --group notebook --extra documents --extra inference --extra discovery-workflow
```

Put `OPENAI_API_KEY` in the repository-root `.env`, select the project kernel and **Run
All**. The first run downloads the PDF as needed and makes paid Luna x-high calls.
Restart in the same workspace to reopen the result. The local stage journal contains
source quotations and provider responses. No live results are saved in this notebook.

In [ ]:
import sys
from pathlib import Path

root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()
)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from domain_kernel import PDF, DocumentSpan
from examples.notebooks.document_discovery import configured_client, open_recruitment_model
from examples.notebooks.originals import load_originals

workspace = root / "local/recruitment-discovery-v23"
originals = root / "local/notebook-tour/recruitment-default-discovery/sources"
original = load_originals(directory=originals, identifiers=["ai-act-en"], download=True)[
    "ai-act-en"
]
client, budget, graph = configured_client(workspace=workspace, originals=originals)
actor = "demo:discovery-reader"
project, model = await open_recruitment_model(client=client, actor=actor)
print(model.info.description)
print(original.digest.value)

## Read the source

Docling groups native document items by structure and token budget. The SDK retains those
chunks and separately addressable page text in SQLite. Read an original page for comparison.

In [ ]:
source = await project.ingest_source(
    name="english-ai-act",
    source=PDF(path=original.relative_path, expected_hash=original.digest),
    actor=actor,
    idempotency_key="ingest-v1",
)
pages = {
    e.locator.page: e
    for e in source.evidence
    if isinstance(e.locator, DocumentSpan) and e.locator.section is None
}
print(f"{len(pages)} physical pages")
print((await project.get_evidence(evidence=pages[127])).excerpt)

## Discover and verify

Pass the revision to the SDK. Discovery uses the model description as its purpose, retrieves
evidence across the PDF, and reviews the draft against that purpose. A failed automated
check can leave a partial draft that is still available to inspect.

In [ ]:
before = budget.usage()
run = await model.discover(
    sources=[source.revision], actor=actor, idempotency_key="discover-recruitment-v23"
)
cases = [case async for case in model.iter_review_cases(run_id=run.id)]
print(f"{len(cases)} review cases · {run.status.value}")
print(f"New provider attempts this execution: {budget.usage().attempts - before.attempts}")

## Inspect the discovered model

Each row is a proposal from this run. Follow the citations to check the meaning, including any
conditions or exceptions. The automated checks do not approve a definition.

In [ ]:
for case in cases:
    for definition in case.definitions:
        print(f"\n{definition.kind.value} · {definition.display_name} · {case.status.value}")
        print(definition.meaning)
    for evidence in case.evidence:
        resolved = await project.get_evidence(evidence=evidence)
        print(f"  {evidence.locator} · {resolved.excerpt}")

In [ ]:
for failure in run.failures:
    print(f"{failure.code}: {failure.message} (retryable: {failure.retryable})")
print("Private response journal:", graph.directory / "schema-stages.sqlite3")